# M2 canonical acquisition
Thin, restart-safe orchestration of reviewed repository code. Gate B remains blocked unless the reviewed target evidence is `confirmed`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
WORKSPACE='/content/drive/MyDrive/giab-wes-nextflow-private'
STAGING='/content/m2-stage'
REPO='/content/giab-wes-nextflow'
REF='feat/m2-data-provenance'  # replace with reviewed commit SHA
RUN_ID='m2-20260903'
from pathlib import Path
assert Path(WORKSPACE).name == 'giab-wes-nextflow-private'
assert not (Path(WORKSPACE)/'DO NOT ACCESS WITH CHATGPT').exists(), 'Safety marker present'


In [ ]:
import shutil, subprocess
if not Path(REPO).exists():
    subprocess.run(['git','clone','https://github.com/jcollins-bioinfo/giab-wes-nextflow.git',REPO],check=True)
subprocess.run(['git','-C',REPO,'fetch','--all','--prune'],check=True)
subprocess.run(['git','-C',REPO,'checkout','--detach',REF],check=True)
repo_sha=subprocess.check_output(['git','-C',REPO,'rev-parse','HEAD'],text=True).strip()
print('repository SHA:',repo_sha)


In [ ]:
import json, os, platform
for path in ['/content',WORKSPACE]:
    usage=shutil.disk_usage(path); print(path,'free_GiB',round(usage.free/2**30,1))
assert shutil.disk_usage('/content').free >= 100*2**30, 'Need >=100 GiB ephemeral space'
runtime={'repository_sha':repo_sha,'platform':platform.platform(),'architecture':platform.machine(),'python':platform.python_version()}
Path(STAGING).mkdir(parents=True,exist_ok=True)
(Path(STAGING)/'runtime.json').write_text(json.dumps(runtime,sort_keys=True,indent=2)+'\n')
print(runtime)


In [ ]:
subprocess.run(['python','-m','pip','install','-r',f'{REPO}/requirements-dev.txt'],check=True)
# The repository driver processes manifest objects sequentially and is safe to rerun.
subprocess.run(['python',f'{REPO}/scripts/acquire_m2.py','--workspace',STAGING,'--run-id',RUN_ID],check=True)


In [ ]:
subprocess.run(['python',f'{REPO}/scripts/prepare_m2.py','--workspace',STAGING,'--run-id',RUN_ID],check=True)
gate=json.loads(Path(f'{REPO}/config/m2-target-design.json').read_text())
if gate['classification']!='confirmed':
    print('Gate A ready; Gate B BLOCKED:',gate['block_reason'])
    print('Recovery: establish exact primary capture-design identity, update reviewed gate record, then rerun preparation with target BED/source dictionary.')
else:
    raise RuntimeError('Provide reviewed --target-bed and --source-dict paths before publication; hidden state is forbidden.')
